In [ ]:

import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)


In [ ]:
import numpy as np
import torch
import os,glob
from collections import OrderedDict
from torchvision import datasets, transforms
from attrdict import AttrDict
from train_v4_map import setup_randomness               

In [ ]:
import os
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"   # see issue #152
os.environ["CUDA_VISIBLE_DEVICES"]="0"


trainset,contexset='cifar10', 'cifar100'



## sets results_path_list 

In [ ]:

results_path_list = [

'./../upload_results/cifar10_cifar100/modelnamefbresnet18st/',

'./../upload_results/cifar10_cifar100/modelnamebmafbresnet18st_paramtrace1/',
'./../upload_results/cifar10_cifar100/modelnamebmafbresnet18st_paramtrace2/',
'./../upload_results/cifar10_cifar100/modelnamebmafbresnet18st_paramtrace3/',

 ]

## evaluation

In [ ]:
from evaluate_v2 import (ts,
                         apply_t,
                         get_acc,
                         get_ll,
                         get_ece,
                         get_ood_auroc,           
                         )

from torch.nn.functional import log_softmax
from sklearn.model_selection import KFold
from scipy.optimize import minimize
from collections import OrderedDict





def eval_trainedmodel(dataloader, model,q_params = None , nfsample=1, topk=3, configs=None):
    assert q_params is not None
    pred_list,label_list,logit_list = [],[],[]    

    model.eval()    
    with torch.no_grad():
        for i, (x, y) in enumerate(dataloader):    
            # predict class            
            outs = model.predict_eval(x.cuda(), nfsample=nfsample, topk=topk)                     

            # compute performance
            logit_list.append(outs.logits.cpu())
            pred_list.append(outs.pred.cpu())
            label_list.append(y.cpu())

            torch.cuda.empty_cache() 
    
    pred_list = torch.cat(pred_list,dim=0)    
    label_list = torch.cat(label_list,dim=0)    
    logit_list = torch.cat(logit_list,dim=1)    
    

        
    # evaluate model
    outs = AttrDict()
    outs.pred_list = pred_list
    outs.label_list = label_list    
    outs.logit_list = logit_list #(ns,nb,npred)
    return outs    






def _compute_diversity_ratio(outs , digits=3):
    logit_list = outs.logit_list.transpose(1,0) #(nb,ns,ndim)
    yhat = logit_list .max(dim=-1)[1]
    nfsample = yhat.shape[-1]

    static_list = []
    for i_yhat in yhat:
        i_yhat_unique,i_yhat_unique_cnt = i_yhat.unique(return_counts=True)
        top1_cnt = i_yhat_unique_cnt.topk(1).values.item()
        majority_cnt = top1_cnt
        not_majority_cnt = nfsample - top1_cnt


        static_list.append( (majority_cnt/nfsample, not_majority_cnt/ nfsample , not_majority_cnt/ majority_cnt) )  #( majority_cnt/nfsample,       not_majority_cnt/ nfsample ,   not_majority_cnt/ majority_cnt )
    
    static_list = np.array(static_list).round(3)
    return static_list.mean(axis=0).round(digits), ((1/np.sqrt(nfsample))*static_list.std(axis=0)).round(digits)








import copy
from evaluate_v2 import (ts,
                         apply_t,
                         get_acc,
                         get_ll,
                         get_ece,
                         get_ood_auroc,      
                         get_ood_auroc2,     
                         get_entropy,              
                         )




def get_results_from_outs(outs,outs_ood1,n_runs=5,n_splits=2,digits=3):
            


    def _compute_logmarginalprob(logits):## new candiate
        #print('logit.shape: {}'.format(logits.shape))
        assert logits.dim() == 3
        ns,ndata,ndim = logits.shape    
        logits   = logits.log_softmax(dim=-1) #(ns,ndata,ndim)
        log_prob = torch.logsumexp(logits, dim=0) - np.log(ns)
        log_prob = log_prob.log_softmax(dim=-1)

        return log_prob.cpu().data.numpy()


    targets          = outs.label_list.cpu().data.numpy()    
    targets_ood1     = outs_ood1.label_list.cpu().data.numpy()
    log_preds        = _compute_logmarginalprob(outs.logit_list)   
    log_ood1_preds   = _compute_logmarginalprob(outs_ood1.logit_list)
    
    
    
    
    
    # ------------------------------------------------
    # evaluate acc,nll, and, ece
    # ------------------------------------------------            
            
    acc            =  get_acc( np.exp(log_preds), targets )
    nll            = -get_ll(  np.exp(log_preds), targets ) 
    ece            =  get_ece(  np.exp(log_preds), targets )    
        
    test_lp_in_out = np.concatenate([log_preds,log_ood1_preds],axis=0)
    ood_labels     = np.concatenate([np.zeros(log_preds.shape[0]),np.ones(log_ood1_preds.shape[0]) ],axis=0)             
    aucroc1        =  get_ood_auroc2(np.exp(test_lp_in_out ), ood_labels  ) 
    
    
    indent1  = get_entropy(  np.exp(log_preds)     )
    outdent1 =  get_entropy( np.exp(log_ood1_preds) )
    

    # evaluate functional diversity
    fdiv_in = _compute_diversity_ratio(outs , digits=3)
    fdiv_out = _compute_diversity_ratio(outs_ood1 , digits=3)
    
    fdivin = fdiv_in[0][1], fdiv_in[1][1]
    fdivout = fdiv_out[0][1], fdiv_out[1][1]
    
    rfdivin = fdiv_in[0][2],fdiv_in[1][2]        
    rfdivout = fdiv_out[0][2],fdiv_out[1][2]
    
    return acc,nll,ece,aucroc1,indent1,outdent1,fdivin,fdivout,rfdivin,rfdivout







#import copy
from torch.nn.functional import log_softmax
from sklearn.model_selection import KFold
from scipy.optimize import minimize
from collections import OrderedDict
from attrdict import AttrDict

def _get_modelspec_from_configs(configs):
    model_spec = configs.model.modelname
    model_spec += '_reglam{}'.format(configs.train.reglambda)
    model_spec += '_topk{}'.format(configs.train.topk) if 'topk' in configs.train.keys() else '_topk{}'.format(10)
    model_spec += '_nepr{}'.format(configs.train.nepochratio) if 'nepochratio' in configs.train.keys() else '_nepr{}'.format(0)
    model_spec += '_mixct{}'.format(configs.train.mixcontext) if 'mixcontext' in configs.train.keys() else  '_mixct{}'.format(False)
    model_spec += '_mixalpha{}'.format(configs.train.mixalpha) if 'mixalpha' in configs.train.keys() else  '_mixmixalpha{}'.format(0.0)
    model_spec += '_paramtrace{}'.format(configs.train.paramtrace) if 'paramtrace' in configs.train.keys() else  '_paramtrace{}'.format(0)    
        
    return model_spec


def update_keys_oddset(configs:AttrDict,ood_dict:dict):
    outs = {**dict(configs.dataset) , **ood_dict}
    configs.dataset = {}
    configs.dataset = AttrDict(outs)
    return copy.deepcopy(configs)


## calibaration revision

In [ ]:
#from attrdict import AttrDict

import os
from addict import Dict as AttrDict 
from datasets_v2 import setup_dataset


trainset,contexset,oodset='cifar10', 'cifar100','svhn'




configs = AttrDict( dataset = AttrDict( trainset = trainset, contextset = contexset, oodset = oodset , ntrainset=45000),
                    train   = AttrDict( nbatch = 1024 ),
                    exp     = AttrDict( seeds = '1111')
                )    

trainloader,valloader,testloader,contextloader,oodloader = setup_dataset(configs,fortrain=False) 




## evaluate the trained models

In [ ]:
from models_deepnet_v4 import _setup_model_deep

def _setup_model_cifar(configs={},formatted='.pth.tar'):              
    model = _setup_model_deep(configs)   
    model = model.cuda() if torch.cuda.is_available() else model
    return model


In [ ]:

def eval_trainedmodel(dataloader, model,q_params = None , use_bayes=False, topk= 10, nfsample=1):
    assert q_params is not None
    pred_list,label_list = [],[]    
    logit_list = []

    model.eval()    
    with torch.no_grad():
        for i, (x, y) in enumerate(dataloader):    
            outs = model.predict_eval(x.cuda(),use_bayes=use_bayes, nfsample=nfsample,topk=topk)                     
                                             
            # compute performance
            pred_list.append(outs.pred.cpu())
            label_list.append(y.cpu())
            logit_list.append(outs.logits.cpu())      


            torch.cuda.empty_cache() 
    
    pred_list = torch.cat(pred_list,dim=0)    
    label_list = torch.cat(label_list,dim=0)    
    logit_list = torch.cat(logit_list,dim=1)    
    
        
    # evaluate model
    outs = AttrDict()
    outs.pred_list = pred_list
    outs.label_list = label_list    
    outs.logit_list = logit_list #(ns,nb,npred)
    return outs    



            
            
formatted                  = '.pth.tar'
topk                       = 10                    
nfsample_list              = [10]
filtered_current_path_list = []

metric_dict = OrderedDict()
for current_path in results_path_list:
    
    current_path_list = sorted(glob.glob(current_path + '/*.pth.tar' ))      
    print('current_path_list: {}'.format(current_path_list))
 

 
    if len(current_path_list) >= 1:
        print('len(current_path_list): {}'.format(len(current_path_list)))
  

        filtered_current_path_list.append(current_path + '_len(file)={}'.format(len(current_path_list)))        
 
        
        for i_th_file in current_path_list:
            
                        
            
            i_th_saved = torch.load(i_th_file)
            print('i_th_file: {}'.format(i_th_file))
            
            ## get model spec     
            modelspec = _get_modelspec_from_configs(i_th_saved['configs'])

            
            ## get saved params        
            setup_randomness(i_th_saved['configs'].exp.seeds)
            model  = _setup_model_cifar(i_th_saved['configs'],formatted =formatted )
            model  = model.cuda()
            model.load_state_dict(i_th_saved['state_dict'])
                

            for nfsample in nfsample_list:
                outs      = eval_trainedmodel(testloader, model, q_params = list(model.parameters()) , use_bayes=True,   topk=topk , nfsample = nfsample)       
                outs_ood1 = eval_trainedmodel(oodloader, model, q_params = list(model.parameters())  , use_bayes=True,   topk=topk , nfsample = nfsample)       
            
                
            

                acc,nll,ece,aucroc1,indent1,outdent1,fdivin,fdivout,rfdivin,rfdivout = get_results_from_outs(outs,outs_ood1)
                        

                # set
                
                model_configs = modelspec + '_nfs{}'.format(nfsample)
                if  model_configs  not in metric_dict.keys():
                    metric_dict[model_configs ] = OrderedDict()
                
            
                if  len(metric_dict[modelspec + '_nfs{}'.format(nfsample)].keys()) == 0 :                
                    metric_dict[model_configs ]['acc'] = [acc]
                    metric_dict[model_configs ]['nll'] = [nll]
                    metric_dict[model_configs ]['ece'] = [ece]
                    metric_dict[model_configs]['auroc'] = [aucroc1]

                        
                else:
                    metric_dict[model_configs]['acc'].append(acc)
                    metric_dict[model_configs]['nll'].append(nll)
                    metric_dict[model_configs]['ece'].append(ece)
                    metric_dict[model_configs]['auroc'].append(aucroc1)

                    
                    


        torch.cuda.empty_cache()
    else:
        print('current_path_list is empty')
    print('\n')
    
    

In [ ]:
labels = ['T-FVI','R-FVI' + r' ($\mathcal{T}_1$)','R-FVI' + r' ($\mathcal{T}_2$)','R-FVI' + r' ($\mathcal{T}_3$)']
colors = ['#1f77b4', '#ff9999' , '#ff6347', '#d62725']

        

## setting

We set training set as CIFAR-10 with context set CIFAR-100 for T-FVI, and set SVHN as OOD set.

For $T=200$, we set the epochs_for_bayes = T*0.7 based on SWAG is constructed after $T*0.75$.

Then, to construce the proposed function-space distribution, we consisder the following trajectory $\mathcal{T}_1,\mathcal{T}_2$ and $\mathcal{T}_3$, as folllows:

    if configs.train.paramtrace == 1:      # $\mathcal{T}_1$ 
        past_periods, period = 20 , 4
        tmp_list = np.arange( epochs_for_bayes - past_periods, epochs_for_bayes) 
        epochs_for_priorconstruction = [i_tmp for i_tmp in tmp_list if i_tmp % period ==0]

    if configs.train.paramtrace == 2:      # $\mathcal{T}_2$
        past_periods, period = 20 , 6
        tmp_list = np.arange( epochs_for_bayes - past_periods, epochs_for_bayes) 
        epochs_for_priorconstruction =[i_tmp for i_tmp in tmp_list if i_tmp % period ==0] 

    if configs.train.paramtrace == 3:       # $\mathcal{T}_3$
        past_periods, period = 20 , 2
        tmp_list = np.arange( epochs_for_bayes - past_periods, epochs_for_bayes) 
        epochs_for_priorconstruction = [i_tmp for i_tmp in tmp_list if i_tmp % period ==0]
 

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
from matplotlib.ticker import PercentFormatter,FormatStrFormatter,ScalarFormatter
from matplotlib.ticker import MaxNLocator



figsiz=(20,3)
metric_list = ['acc','nll','ece','auroc']
arrow_list  = [r'$\uparrow$',r'$\downarrow$',r'$\downarrow$',r'$\uparrow$']
f,axs= plt.subplots( ncols=4,  figsize=figsiz )    

for k,metric in enumerate(metric_list):
    metric_result = np.array([ (np.array(i_th_model[metric]).mean(),np.array(i_th_model[metric]).std()) for i_th_model in metric_dict.values()]).squeeze()
    #print(metric_result)


    mean    = metric_result[:,0]
    stderr  = metric_result[:,1]/np.sqrt(3)

    xrange = np.arange(1,len(metric_result)+1)       
    axs[k].bar(xrange , mean, yerr=stderr , width=0.75,
        linewidth=2.,
        color=colors,
        edgecolor='black',
        alpha=0.75,
         capsize=5,
    )    

    plt.tick_params(axis='both', labelsize=13)
    axs[k].set_xticks(xrange, labels,fontsize=13)
    axs[k].set_ylabel(metric,fontsize=13)

    max_  = max(mean)
    min_  = min(mean)
    delta= 0.5*(max_ - min_)
    axs[k].set_ylim(min_-delta, max_+ delta)
    axs[k].yaxis.set_major_locator(MaxNLocator(nbins=5))  # Max 5 y-ticks

    axs[k].set_title(" ${}$({}) across different trajectory ".format(metric,arrow_list[k]),fontsize=13)
    axs[k].grid(True, axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()
